In [1]:
import json
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

all_chunks = []
with open("../data/processed/kb_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        all_chunks.append(json.loads(line))
print("Loaded chunks:", len(all_chunks))

def load_pairs(name):
    pairs = []
    with open(f"../data/processed/qa_pairs_{name}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            pairs.append(json.loads(line))
    return pairs

train_pairs = load_pairs("train")
dev_pairs = load_pairs("dev")
test_pairs = load_pairs("test")
print(f"Pairs: train={len(train_pairs)}, dev={len(dev_pairs)}, test={len(test_pairs)}")

retriever = SentenceTransformer("../data/indexes/retriever_finetuned_v1")
reranker = CrossEncoder("../data/indexes/reranker_finetuned_v1")
print("Retriever device:", retriever.device)

chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]
chunk_text_by_id = {c["chunk_id"]: c["text"] for c in all_chunks}
chunk_embeddings = retriever.encode(
    chunk_texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print("Chunk embeddings shape:", chunk_embeddings.shape)

Loaded chunks: 4781
Pairs: train=19640, dev=2461, test=2468


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retriever device: cuda:0


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Chunk embeddings shape: (4781, 384)


In [2]:
from datasets import load_dataset

In [3]:
dial_data = load_dataset("parquet", data_files="hf://datasets/IBM/multidoc2dial@~parquet/dialogue_domain/train/*.parquet")
print(dial_data)

no_ref_examples = []
for dial in dial_data["train"]:
    turns = dial["turns"]
    for i, turn in enumerate(turns):
        if turn["role"] != "agent" or turn["references"]:
            continue
        prev_user_turns = [t for t in turns[:i] if t["role"] == "user"]
        if not prev_user_turns:
            continue
        no_ref_examples.append({
            "domain": dial["domain"],
            "user_query": prev_user_turns[-1]["utterance"],
            "agent_response": turn["utterance"],
        })

print("Agent turns with NO references (candidate ESCALATE examples):", len(no_ref_examples))
print()
for ex in no_ref_examples[:15]:
    print("USER:", ex["user_query"][:100])
    print("AGENT:", ex["agent_response"][:150])
    print("---")

DatasetDict({
    train: Dataset({
        features: ['dial_id', 'domain', 'turns'],
        num_rows: 3474
    })
})
Agent turns with NO references (candidate ESCALATE examples): 0



In [4]:
total_agent_turns = 0
agent_turns_with_refs = 0
agent_turns_without_refs = 0

for dial in dial_data["train"]:
    for turn in dial["turns"]:
        if turn["role"] != "agent":
            continue
        total_agent_turns += 1
        if turn["references"]:
            agent_turns_with_refs += 1
        else:
            agent_turns_without_refs += 1

print("Total agent turns:", total_agent_turns)
print("With references:", agent_turns_with_refs)
print("Without references:", agent_turns_without_refs)

Total agent turns: 24603
With references: 24603
Without references: 0


In [5]:
def get_gold_rank_and_score(query, gold_chunk_ids, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20):
    query_emb = retriever.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = (query_emb @ chunk_embeddings.T)[0]
    top_idx = np.argsort(-sims)[:k]
    candidates = [chunk_ids[i] for i in top_idx]

    pairs = [[query, chunk_text_by_id[cid]] for cid in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    order = np.argsort(-scores)
    reranked_candidates = [candidates[i] for i in order]
    reranked_scores = [scores[i] for i in order]

    gold_set = set(gold_chunk_ids)
    for rank, (cid, score) in enumerate(zip(reranked_candidates, reranked_scores), start=1):
        if cid in gold_set:
            return rank, float(score)
    return None, None

import random
random.seed(0)
sample_pairs = random.sample(train_pairs, 40)

results = []
for p in sample_pairs:
    rank, score = get_gold_rank_and_score(p["query"], p["gold_chunk_ids"], retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)
    results.append((p["query"], rank, score))

results.sort(key=lambda x: (x[1] is None, x[1] if x[1] else 999))
for query, rank, score in results:
    print(f"rank={rank}, score={score}: {query[:100]}")

rank=1, score=0.4294349253177643: agent: Unfortunately, no relevant information is found.
user: es posible obtener un COE?
agent: if y
rank=1, score=0.14581608772277832: user: Hello, I would like to obtain from How Financial Aid Works

rank=1, score=-1.162324070930481: user: Can I use a birth certificate as proof of identity for my child?
agent: No, social security ne
rank=1, score=0.6416015625: user: what is SSI
rank=1, score=0.06879885494709015: user: Regarding the basic eligibility criteria, what should I keep in mind before signing the certif
rank=1, score=1.1999186277389526: user: I need help with changing my address. Can you walk me through the process?
agent: Of course. D
rank=1, score=-0.0746602788567543: user: hi i want to apply for the original card for a non-citizen child
agent: In general, only non-c
rank=1, score=0.38925936818122864: user: About requesting or paying for TVB tickets, on what theme or topic is the information based?
a
rank=1, score=-1.795459270477295: user: 

In [6]:
def find_and_print_context(fragment_text, n_before=3):
    for dial in dial_data["train"]:
        turns = dial["turns"]
        for i, turn in enumerate(turns):
            if turn["role"] == "user" and turn["utterance"].strip() == fragment_text.strip():
                print(f"=== Found in dialogue {dial['dial_id']} (domain={dial['domain']}) ===")
                start = max(0, i - n_before)
                for t in turns[start:i+2]:
                    print(f"  {t['role']}: {t['utterance'][:150]}")
                print()
                return
    print("Not found (may be truncated text) — try a shorter/exact match")

find_and_print_context("No, not yet.")
find_and_print_context("Yes, I do.")
find_and_print_context("there are more details of DMV")

=== Found in dialogue 8042966ce663c4470857062f230c1155 (domain=dmv) ===
  user: I would like to know what would happen if I forget not to respond to the notice of non-compliance and licence downgrade.
  agent: That will result in your Commercial Driver License automatically being downgraded to a Non-CDL class and/or your Commercial Learner Permit being cance
  agent: Has your license already been downgraded?
  user: No, not yet.
  agent: Do you hold a Commercial Learner Permit?

=== Found in dialogue 89b0a713771066b38c48080368de35a1 (domain=dmv) ===
  user: Another question for you regarding vehicle registration renewal, can I renew my registration by mail?
  agent: Absolutely. You'll need to mail the bottom part of your registration renewal notice MV-3 or OP-3 to the address listed on that notice, as well as a pe
  agent: Do you have your MV-3 or OP-3?
  user: Yes, I do.
  agent: Excellent. For future reference, you would need to complete the Application for Duplicate/Renewal Registra

In [7]:
import random
random.seed(1)

sample_pairs = random.sample(train_pairs, 1500)

def batch_score_top1(pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20):
    queries = [p["query"] for p in pairs]
    query_embeddings = retriever.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i, p in enumerate(pairs):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([p["query"], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    results = []
    pos = 0
    for i, p in enumerate(pairs):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        order = np.argsort(-scores)
        top1_idx = order[0]
        top1_chunk = candidates[top1_idx]
        top1_score = float(scores[top1_idx])
        is_correct = top1_chunk in set(p["gold_chunk_ids"])
        results.append({"query": p["query"], "top1_score": top1_score, "top1_correct": is_correct})
    return results

results = batch_score_top1(sample_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

correct_scores = [r["top1_score"] for r in results if r["top1_correct"]]
wrong_scores = [r["top1_score"] for r in results if not r["top1_correct"]]

print(f"Sample size: {len(results)}")
print(f"Top-1 correct: {len(correct_scores)} ({len(correct_scores)/len(results):.1%})")
print(f"Top-1 wrong: {len(wrong_scores)} ({len(wrong_scores)/len(results):.1%})")
print()
print("Score distribution when top-1 is CORRECT:")
print(f"  mean={np.mean(correct_scores):.3f}, median={np.median(correct_scores):.3f}, "
      f"p10={np.percentile(correct_scores,10):.3f}, p25={np.percentile(correct_scores,25):.3f}")
print()
print("Score distribution when top-1 is WRONG:")
print(f"  mean={np.mean(wrong_scores):.3f}, median={np.median(wrong_scores):.3f}, "
      f"p75={np.percentile(wrong_scores,75):.3f}, p90={np.percentile(wrong_scores,90):.3f}")

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Sample size: 1500
Top-1 correct: 445 (29.7%)
Top-1 wrong: 1055 (70.3%)

Score distribution when top-1 is CORRECT:
  mean=0.498, median=0.449, p10=-0.978, p25=-0.274

Score distribution when top-1 is WRONG:
  mean=-0.158, median=-0.187, p75=0.533, p90=1.174


In [8]:
def batch_score_topk_check(pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20, check_k=3):
    queries = [p["query"] for p in pairs]
    query_embeddings = retriever.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i, p in enumerate(pairs):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([p["query"], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    results = []
    pos = 0
    for i, p in enumerate(pairs):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        order = np.argsort(-scores)
        ranked_candidates = [candidates[j] for j in order]
        ranked_scores = [float(scores[j]) for j in order]

        gold_set = set(p["gold_chunk_ids"])
        top1_correct = ranked_candidates[0] in gold_set
        topk_correct = any(cid in gold_set for cid in ranked_candidates[:check_k])
        top1_score = ranked_scores[0]
        margin = ranked_scores[0] - ranked_scores[1]  # gap between best and second-best

        results.append({
            "top1_score": top1_score, "margin": margin,
            "top1_correct": top1_correct, "topk_correct": topk_correct,
        })
    return results

results2 = batch_score_topk_check(sample_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, check_k=3)

topk_correct_n = sum(r["topk_correct"] for r in results2)
print(f"Top-1 correct: {sum(r['top1_correct'] for r in results2)} ({sum(r['top1_correct'] for r in results2)/len(results2):.1%})")
print(f"Gold in top-3: {topk_correct_n} ({topk_correct_n/len(results2):.1%})")

correct_margin = [r["margin"] for r in results2 if r["top1_correct"]]
wrong_margin = [r["margin"] for r in results2 if not r["top1_correct"]]
print()
print(f"Margin (top1 - top2 score) when CORRECT: mean={np.mean(correct_margin):.3f}, median={np.median(correct_margin):.3f}")
print(f"Margin (top1 - top2 score) when WRONG:   mean={np.mean(wrong_margin):.3f}, median={np.median(wrong_margin):.3f}")

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Top-1 correct: 445 (29.7%)
Gold in top-3: 698 (46.5%)

Margin (top1 - top2 score) when CORRECT: mean=1.306, median=0.945
Margin (top1 - top2 score) when WRONG:   mean=0.654, median=0.263


In [9]:
def build_triage_features(pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20, answer_k=3):
    queries = [p["query"] for p in pairs]
    query_embeddings = retriever.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i in range(len(pairs)):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([queries[i], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    examples = []
    pos = 0
    for i, p in enumerate(pairs):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        retriever_top1_sim = float(sims[i][top_idx[i][0]])

        order = np.argsort(-scores)
        ranked_candidates = [candidates[j] for j in order]
        ranked_scores = [float(scores[j]) for j in order]

        gold_set = set(p["gold_chunk_ids"])
        gold_in_topk = any(cid in gold_set for cid in ranked_candidates[:answer_k])

        examples.append({
            "query": p["query"],
            "domain": p["domain"],
            "retriever_top1_sim": retriever_top1_sim,
            "rerank_top1_score": ranked_scores[0],
            "rerank_top3_mean": float(np.mean(ranked_scores[:3])),
            "margin": ranked_scores[0] - ranked_scores[1],
            "label": "ANSWER" if gold_in_topk else "ESCALATE",
        })
    return examples

triage_train_answer_escalate = build_triage_features(train_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)
triage_dev_answer_escalate = build_triage_features(dev_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

from collections import Counter
print("Train label distribution:", Counter(e["label"] for e in triage_train_answer_escalate))
print("Dev label distribution:", Counter(e["label"] for e in triage_dev_answer_escalate))
print()
print("Example:", triage_train_answer_escalate[0])

Batches:   0%|          | 0/307 [00:00<?, ?it/s]

Batches:   0%|          | 0/6138 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/770 [00:00<?, ?it/s]

Train label distribution: Counter({'ESCALATE': 10196, 'ANSWER': 9444})
Dev label distribution: Counter({'ESCALATE': 1287, 'ANSWER': 1174})

Example: {'query': 'user: Hello, I forgot o update my address, can you help me with that?', 'domain': 'dmv', 'retriever_top1_sim': 0.5655285716056824, 'rerank_top1_score': 0.04153377190232277, 'rerank_top3_mean': -0.41183123861749965, 'margin': 0.11474770680069923, 'label': 'ANSWER'}


In [11]:
from datasets import load_dataset

squad_data = load_dataset("rajpurkar/squad")
print(squad_data)

sample = squad_data["train"][0]
print("\nFields:", sample.keys())
for k, v in sample.items():
    print(f"\n{k}:")
    print(str(v)[:300])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

c:\Users\Shrey\Desktop\TUF_F15_Data\Shrey_academics\Deep_Learning\Project_DL\Project_Attempt2\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shrey\.cache\huggingface\hub\datasets--rajpurkar--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Fields: dict_keys(['id', 'title', 'context', 'question', 'answers'])

id:
5733be284776f41900661182

title:
University_of_Notre_Dame

context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 

question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

answers:
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [12]:
import random
random.seed(7)

squad_questions = list(set(squad_data["train"]["question"]))  # dedupe (SQuAD has repeats across paragraphs of the same article)
sampled_reject_questions = random.sample(squad_questions, 10000)

print("Sampled SQuAD questions:", len(sampled_reject_questions))
print("Example:", sampled_reject_questions[0])

Sampled SQuAD questions: 10000
Example: What anniversary did the Arena Football League celebrate in 2012?


In [13]:
def build_reject_features(questions, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20):
    query_embeddings = retriever.encode(questions, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i in range(len(questions)):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([questions[i], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    examples = []
    pos = 0
    for i, q in enumerate(questions):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        retriever_top1_sim = float(sims[i][top_idx[i][0]])
        order = np.argsort(-scores)
        ranked_scores = [float(scores[j]) for j in order]

        examples.append({
            "query": q,
            "domain": "out_of_domain",
            "retriever_top1_sim": retriever_top1_sim,
            "rerank_top1_score": ranked_scores[0],
            "rerank_top3_mean": float(np.mean(ranked_scores[:3])),
            "margin": ranked_scores[0] - ranked_scores[1],
            "label": "REJECT",
        })
    return examples

# split the 10k SQuAD sample into train/dev the same 90/10-ish way, to match our other splits
random.shuffle(sampled_reject_questions)
split_point = int(0.9 * len(sampled_reject_questions))
squad_train_q = sampled_reject_questions[:split_point]
squad_dev_q = sampled_reject_questions[split_point:]

triage_train_reject = build_reject_features(squad_train_q, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)
triage_dev_reject = build_reject_features(squad_dev_q, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

print("Train REJECT examples:", len(triage_train_reject))
print("Dev REJECT examples:", len(triage_dev_reject))

# sanity check: compare REJECT confidence distribution to ANSWER/ESCALATE from earlier
reject_scores = [e["rerank_top1_score"] for e in triage_train_reject]
answer_scores = [e["rerank_top1_score"] for e in triage_train_answer_escalate if e["label"] == "ANSWER"]
escalate_scores = [e["rerank_top1_score"] for e in triage_train_answer_escalate if e["label"] == "ESCALATE"]

print(f"\nREJECT   top1_score: mean={np.mean(reject_scores):.3f}, median={np.median(reject_scores):.3f}")
print(f"ANSWER   top1_score: mean={np.mean(answer_scores):.3f}, median={np.median(answer_scores):.3f}")
print(f"ESCALATE top1_score: mean={np.mean(escalate_scores):.3f}, median={np.median(escalate_scores):.3f}")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/2813 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Train REJECT examples: 9000
Dev REJECT examples: 1000

REJECT   top1_score: mean=-0.957, median=-1.021
ANSWER   top1_score: mean=0.352, median=0.300
ESCALATE top1_score: mean=-0.225, median=-0.273


In [14]:
triage_train_all = triage_train_answer_escalate + triage_train_reject
triage_dev_all = triage_dev_answer_escalate + triage_dev_reject

random.shuffle(triage_train_all)
random.shuffle(triage_dev_all)

print("Train label distribution:", Counter(e["label"] for e in triage_train_all))
print("Dev label distribution:", Counter(e["label"] for e in triage_dev_all))

with open("../data/processed/triage_train.jsonl", "w", encoding="utf-8") as f:
    for e in triage_train_all:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

with open("../data/processed/triage_dev.jsonl", "w", encoding="utf-8") as f:
    for e in triage_dev_all:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("Saved triage_train.jsonl and triage_dev.jsonl")

Train label distribution: Counter({'ESCALATE': 10196, 'ANSWER': 9444, 'REJECT': 9000})
Dev label distribution: Counter({'ESCALATE': 1287, 'ANSWER': 1174, 'REJECT': 1000})
Saved triage_train.jsonl and triage_dev.jsonl
